<a href="https://colab.research.google.com/github/Naveed101633/Retrieval-Augmented-Generation-RAG-Learning-Lab/blob/main/information_retrieval_search_foundation/Vector_Embeddings_RAG_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vector Embeddings in RAG

## Lab Overview

In this lab, you will explore **vector embeddings** using an embedding model. This is a foundational concept for building Retrieval-Augmented Generation (RAG) systems.

### Learning Objectives

By the end of this lab, you will:

1. ✅ Understand how to use vector embeddings to find and understand contextual information
2. ✅ Learn the basics of **cosine similarity** and **Euclidean distance** for comparing embeddings
3. ✅ Implement embeddings with a transformer-based model
4. ✅ Visualize high-dimensional embeddings using PCA

---

## Table of Contents

1. [Introduction](#1---introduction)
   - 1.1 [A Bit of Theory](#11-a-bit-of-theory)
   - 1.2 [The Framework](#12-the-framework)
2. [The Embedding Model](#2---the-embedding-model)
   - 2.1 [Introduction](#21-introduction)
   - 2.2 [Loading the Model](#22-loading-the-model)
   - 2.3 [Embeddings in Practice](#23-embeddings-in-practice)
   - 2.4 [Visualizing Word Embeddings](#24-visualizing-word-embeddings)
3. [Embeddings and Input Size](#3---embeddings-and-input-size)
   - 3.1 [An Example](#31-an-example)
4. [Conclusion](#4---conclusion)

---

## 🔧 Environment Setup

First, let's install all required dependencies and set up our environment.

In [ ]:
# Install required packages
!pip install -q sentence-transformers
!pip install -q torch torchvision
!pip install -q numpy matplotlib scikit-learn ipywidgets

print("✅ All dependencies installed successfully!")

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("✅ Libraries imported successfully!")

---

# 1 - Introduction

In the context of **Retrieval-Augmented Generation (RAG)**, vector embeddings serve as the backbone for intelligent information retrieval. Let's understand their role:

## 🎯 Why Vector Embeddings?

Vector embeddings are used for:

### 1. **Powering Search**

**Capturing Meaning:** Vector embeddings act like a map for text. They convert words and sentences into positions in vector space that capture semantic meaning. These vectors can then be used to locate information matching a query.

**Comparing Similarity:** When a prompt is received, it is converted into an embedding vector of its own. Then, the similarity between this prompt's vector and other vectors in the database can be calculated. This helps identify texts closest in meaning to the prompt.

### 2. **Understanding Context**

**Context Matters:** They help in understanding the context of words in a query, ensuring that the best-matched information is found.

**Flexibility:** Contextual embeddings allow for adapting to different meanings and capturing details that might otherwise be overlooked.

### 💡 Key Insight

In essence, vector embeddings are a behind-the-scenes technology that facilitates smarter, more helpful, and accurate data retrieval by capturing nuances in a way that no other search technique provides.

## 1.1 A Bit of Theory

In vector retrieval, you use an **embedding model** to convert prompts and documents into vectors. To find the most relevant documents for a query, you check their similarity using distance measures:

### 📐 Cosine Similarity

This evaluates how close two vectors are based on their **angle**. For a query vector $\mathbf{q}$ and a document vector $\mathbf{d}$:

$$
\text{cosine\_similarity}(\mathbf{q}, \mathbf{d}) = \frac{\mathbf{q} \cdot \mathbf{d}}{\|\mathbf{q}\| \|\mathbf{d}\|}
$$

**Interpretation:** A value close to **1** indicates that the vectors, and thus the texts, are very similar. A value close to **0** indicates they are unrelated.

### 📏 Euclidean Distance

This calculates the **"straight-line"** distance between two vectors in the embedding space:

$$
\text{euclidean\_distance}(\mathbf{q}, \mathbf{d}) = \sqrt{\sum_{i=1}^{n} (q_i - d_i)^2}
$$

**Interpretation:** **Smaller** distances suggest more closely related content.

### ⚠️ Important Note

Be mindful that for some metrics:
- **Higher score** indicates greater similarity (e.g., cosine similarity)
- **Lower score** indicates greater similarity (e.g., Euclidean distance)

## 1.2 The Framework

Here's the general workflow for using embeddings in retrieval:

```
┌─────────────────────────────────────────────────────────┐
│  1. Create the Embedding                                │
│     Use an embedding model to convert query and         │
│     documents into vectors                              │
└─────────────────────────────────────────────────────────┘
                        ↓
┌─────────────────────────────────────────────────────────┐
│  2. Metric Measurement                                  │
│     Use a similarity metric to determine how close      │
│     each document is to your query                      │
└─────────────────────────────────────────────────────────┘
                        ↓
┌─────────────────────────────────────────────────────────┐
│  3. Sorting                                             │
│     Sort documents by similarity score and select       │
│     the top few as the most relevant                    │
└─────────────────────────────────────────────────────────┘
```

This gives us an easy way to query a database! Let's explore! 🚀

## 📊 Implementing Distance Formulas

Now let's implement the distance formulas from scratch to understand how they work.

In [ ]:
def cosine_similarity(v1, array_of_vectors):
    """
    Compute the cosine similarity between a vector and an array of vectors.

    Cosine similarity measures the cosine of the angle between two vectors,
    which indicates how similar they are in direction (not magnitude).

    Parameters:
    -----------
    v1 : array-like
        The first vector (query vector)
    array_of_vectors : array-like
        An array of vectors or a single vector (document vectors)

    Returns:
    --------
    list
        A list of cosine similarities between v1 and each vector in array_of_vectors.
        Values range from -1 to 1, where 1 means identical direction.

    Example:
    --------
    >>> v1 = [1, 0]
    >>> v2 = [1, 1]
    >>> cosine_similarity(v1, v2)
    [0.7071067811865475]  # cos(45°) ≈ 0.707
    """
     # Ensure that v1 is a numpy array
    v1 = np.array(v1)

    # Initialize a list to store similarities
    similarities = []

    # Check if array_of_vectors is a single vector
    if len(np.shape(array_of_vectors)) == 1:
        array_of_vectors = [array_of_vectors]

    # Iterate over each vector in the array
    for v2 in array_of_vectors:
        # Convert the current vector to a numpy array
        v2 = np.array(v2)

        # Compute the dot product of v1 and v2
        dot_product = np.dot(v1, v2)

        # Compute the norms (magnitudes) of the vectors
        norm_v1 = np.linalg.norm(v1)
        norm_v2 = np.linalg.norm(v2)

        # Compute the cosine similarity and append to the list
        similarity = dot_product / (norm_v1 * norm_v2)
        similarities.append(similarity)

    return [float(x) for x in similarities]

print("✅ Cosine Similarity function defined!")


In [ ]:
def euclidean_distance(v1, array_of_vectors):
    """
    Compute the Euclidean distance between a vector and an array of vectors.

    Euclidean distance is the straight-line distance between two points in space.
    It measures the actual geometric distance between vectors.

    Parameters:
    -----------
    v1 : array-like
        The first vector (query vector)
    array_of_vectors : array-like
        An array of vectors or a single vector (document vectors)

    Returns:
    --------
    list
        A list of Euclidean distances between v1 and each vector in array_of_vectors.
        Lower values indicate closer (more similar) vectors.

    Example:
    --------
    >>> v1 = [0, 0]
    >>> v2 = [3, 4]
    >>> euclidean_distance(v1, v2)
    [5.0]  # sqrt(3² + 4²) = 5
    """

     # Ensure that v1 is a numpy array
    v1 = np.array(v1)

    # Initialize a list to store distances
    distances = []

    # Check if array_of_vectors is a single vector
    if len(np.shape(array_of_vectors)) == 1:
        array_of_vectors = [array_of_vectors]

    # Iterate over each vector in the array
    for v2 in array_of_vectors:
        # Convert the current vector to a numpy array
        v2 = np.array(v2)

        # Check if the input arrays have the same shape
        if v1.shape != v2.shape:
            raise ValueError(f"Shapes don't match: v1 shape: {v1.shape}, v2 shape: {v2.shape}")

        # Calculate the Euclidean distance and append to the list
        # Formula: sqrt(sum((v1_i - v2_i)²))
        dist = np.sqrt(np.sum((v1 - v2) ** 2))
        distances.append(dist)

    return [float(x) for x in distances]

print("✅ Euclidean Distance function defined!")



## 🧪 Testing the Distance Functions

Let's test our implementations with simple 2D vectors to understand how these metrics work.

In [ ]:
# Define example vectors
v1 = [1, 2]  # Our query vector
v2 = [1, 1]  # A single document vector
array_v = [[3, 2], [5, 6]]  # Multiple document vectors

# Calculate cosine similarity
cosine_v1_v2 = cosine_similarity(v1, v2)
cosine_v1_array_v = cosine_similarity(v1, array_v)

# Calculate Euclidean distance
euclidean_v1_v2 = euclidean_distance(v1, v2)
euclidean_v1_array_v = euclidean_distance(v1, array_v)

# Display results
print("=" * 60)
print("DISTANCE METRIC COMPARISON")
print("=" * 60)
print(f"\n📍 Query Vector (v1): {v1}")
print(f"📍 Document Vector (v2): {v2}")
print(f"📍 Document Vectors Array: {array_v}")
print("\n" + "-" * 60)
print("COSINE SIMILARITY (Higher = More Similar)")
print("-" * 60)
print(f"Between v1 and v2: {cosine_v1_v2[0]:.4f}")
print(f"Between v1 and array_v[0]: {cosine_v1_array_v[0]:.4f}")
print(f"Between v1 and array_v[1]: {cosine_v1_array_v[1]:.4f}")
print("\n" + "-" * 60)
print("EUCLIDEAN DISTANCE (Lower = More Similar)")
print("-" * 60)
print(f"Between v1 and v2: {euclidean_v1_v2[0]:.4f}")
print(f"Between v1 and array_v[0]: {euclidean_v1_array_v[0]:.4f}")
print(f"Between v1 and array_v[1]: {euclidean_v1_array_v[1]:.4f}")
print("\n" + "=" * 60)

### 🔍 Key Observation

Notice that the output is always a **list**. Observe the following:

- **In terms of cosine similarity**, the vector closest to `v1` is the **second vector** in the array `[5, 6]`
- **In terms of Euclidean distance**, the vector closest to `v1` is the **first vector** in the array `[3, 2]`

#### Why the difference?

This occurs because these metrics evaluate **different attributes**:

- **Cosine Similarity** measures the **angle** between two vectors (direction)
- **Euclidean Distance** measures the **actual distance** between them (magnitude)

Consequently, with cosine similarity, the actual distances between the points forming the vectors are irrelevant. What matters is their direction!

Let's visualize this to better understand! 📊

## 📈 Visualizing Vector Similarities

In [ ]:
def plot_vectors():
    """
    Visualize vectors and their relationships in 2D space.
    This helps understand the difference between cosine similarity and Euclidean distance.
    """
    # Define vectors
    v1 = np.array([1, 2])
    v2 = np.array([1, 1])
    v3 = np.array([3, 2])
    v4 = np.array([5, 6])

    # Create figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # ===== Left Plot: Cosine Similarity Visualization =====
    ax1.set_title('Cosine Similarity (Angle-based)', fontsize=14, fontweight='bold')
    ax1.set_xlabel('X', fontsize=12)
    ax1.set_ylabel('Y', fontsize=12)
    ax1.grid(True, alpha=0.3)
    ax1.axhline(y=0, color='k', linewidth=0.5)
    ax1.axvline(x=0, color='k', linewidth=0.5)

    # Plot vectors as arrows
    ax1.arrow(0, 0, v1[0], v1[1], head_width=0.3, head_length=0.3, fc='blue', ec='blue', linewidth=2, label='v1 [1,2] (Query)')
    ax1.arrow(0, 0, v2[0], v2[1], head_width=0.3, head_length=0.3, fc='green', ec='green', linewidth=2, label='v2 [1,1]')
    ax1.arrow(0, 0, v3[0], v3[1], head_width=0.3, head_length=0.3, fc='orange', ec='orange', linewidth=2, label='v3 [3,2]')
    ax1.arrow(0, 0, v4[0], v4[1], head_width=0.3, head_length=0.3, fc='red', ec='red', linewidth=2, label='v4 [5,6]')

    # Calculate and display cosine similarities
    cos_sim_v2 = cosine_similarity(v1, v2)[0]
    cos_sim_v3 = cosine_similarity(v1, v3)[0]
    cos_sim_v4 = cosine_similarity(v1, v4)[0]

    ax1.text(1.5, 2.5, f'Similarity v1-v2: {cos_sim_v2:.3f}', fontsize=10, bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))
    ax1.text(3.5, 2.5, f'Similarity v1-v3: {cos_sim_v3:.3f}', fontsize=10, bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    ax1.text(5.5, 6.5, f'Similarity v1-v4: {cos_sim_v4:.3f} ★', fontsize=10, fontweight='bold', bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.8))

    ax1.legend(loc='upper left', fontsize=10)
    ax1.set_xlim(-1, 7)
    ax1.set_ylim(-1, 8)
    ax1.text(3.5, -0.7, 'v4 has highest cosine similarity (similar direction)',
             fontsize=11, ha='center', style='italic', bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

    # ===== Right Plot: Euclidean Distance Visualization =====
    ax2.set_title('Euclidean Distance (Distance-based)', fontsize=14, fontweight='bold')
    ax2.set_xlabel('X', fontsize=12)
    ax2.set_ylabel('Y', fontsize=12)
    ax2.grid(True, alpha=0.3)
    ax2.axhline(y=0, color='k', linewidth=0.5)
    ax2.axvline(x=0, color='k', linewidth=0.5)

    # Plot vectors as arrows
    ax2.arrow(0, 0, v1[0], v1[1], head_width=0.3, head_length=0.3, fc='blue', ec='blue', linewidth=2, label='v1 [1,2] (Query)')
    ax2.arrow(0, 0, v2[0], v2[1], head_width=0.3, head_length=0.3, fc='green', ec='green', linewidth=2, label='v2 [1,1]')
    ax2.arrow(0, 0, v3[0], v3[1], head_width=0.3, head_length=0.3, fc='orange', ec='orange', linewidth=2, label='v3 [3,2]')
    ax2.arrow(0, 0, v4[0], v4[1], head_width=0.3, head_length=0.3, fc='red', ec='red', linewidth=2, label='v4 [5,6]')

    # Draw distance lines from v1 to other vectors
    ax2.plot([v1[0], v2[0]], [v1[1], v2[1]], 'g--', linewidth=1.5, alpha=0.7)
    ax2.plot([v1[0], v3[0]], [v1[1], v3[1]], 'orange', linestyle='--', linewidth=1.5, alpha=0.7)
    ax2.plot([v1[0], v4[0]], [v1[1], v4[1]], 'r--', linewidth=1.5, alpha=0.7)

    # Calculate and display Euclidean distances
    euc_dist_v2 = euclidean_distance(v1, v2)[0]
    euc_dist_v3 = euclidean_distance(v1, v3)[0]
    euc_dist_v4 = euclidean_distance(v1, v4)[0]

    ax2.text(1.5, 2.5, f'Distance v1-v2: {euc_dist_v2:.3f}', fontsize=10, bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))
    ax2.text(3.5, 2.5, f'Distance v1-v3: {euc_dist_v3:.3f} ★', fontsize=10, fontweight='bold', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    ax2.text(5.5, 6.5, f'Distance v1-v4: {euc_dist_v4:.3f}', fontsize=10, bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.8))

    ax2.legend(loc='upper left', fontsize=10)
    ax2.set_xlim(-1, 7)
    ax2.set_ylim(-1, 8)
    ax2.text(3.5, -0.7, 'v3 has smallest Euclidean distance (closest point)',
             fontsize=11, ha='center', style='italic', bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

    plt.tight_layout()
    plt.show()

    # Print summary
    print("\n" + "=" * 70)
    print("VISUALIZATION SUMMARY")
    print("=" * 70)
    print("\n🎯 COSINE SIMILARITY (Left Plot):")
    print(f"   Most similar to v1: v4 [5,6] with similarity {cos_sim_v4:.4f}")
    print("   → Focuses on DIRECTION (angle between vectors)")
    print("\n📏 EUCLIDEAN DISTANCE (Right Plot):")
    print(f"   Closest to v1: v3 [3,2] with distance {euc_dist_v3:.4f}")
    print("   → Focuses on MAGNITUDE (actual distance between points)")
    print("\n💡 Key Takeaway:")
    print("   Different metrics capture different aspects of similarity!")
    print("=" * 70)

print("✅ Visualization function defined!")

In [ ]:
# Visualize the vectors
plot_vectors()

---

# 2 - The Embedding Model

Now that we understand the theory, let's work with real embedding models!

## 2.1 Introduction

The **embedding model** is responsible for converting a word or sentence into a **fixed-size vector**.

### Key Characteristics:

- ✅ Trained on **millions of samples**
- ✅ Specialized in grouping **semantically related** sentences
- ✅ Produces **consistent vector representations**

### How It Works:

```
Input Text → Embedding Model → Vector Representation

"The cat sat on the mat"  →  [0.23, -0.45, 0.67, ..., 0.12]
                                    (768 dimensions)
```

Similar sentences will have similar vectors in the embedding space!

## 2.2 Loading the Model

We'll use the **BAAI/bge-base-en-v1.5** model:

- 🏆 High-quality transformer-based model
- 📊 Generates **768-dimensional** embeddings
- 🌍 Optimized for English text
- ⚡ Efficient and widely used in production

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the embedding model
print("Loading the embedding model...")
print("This may take a minute on first run (downloading model weights)...\n")

model = SentenceTransformer('BAAI/bge-base-en-v1.5')

print("✅ Model loaded successfully!")
print(f"\n📊 Model Details:")
print(f"   - Name: BAAI/bge-base-en-v1.5")
print(f"   - Embedding Dimension: 768")
print(f"   - Max Sequence Length: 512 tokens")
print(f"   - Type: Transformer-based sentence embedding model")

## 2.3 Embeddings in Practice

Let's see how the model creates embeddings for different sentences and compare their similarities!

In [ ]:
# Define example sentences
sentences = [
    "The cat is sleeping on the couch.",
    "A feline is resting on the sofa.",
    "The stock market crashed today.",
    "I love eating pizza for dinner.",
    "Machine learning is a subset of artificial intelligence."
]

# Generate embeddings
print("Generating embeddings for example sentences...\n")
embeddings = model.encode(sentences)

# Display information
print("=" * 70)
print("EMBEDDING GENERATION RESULTS")
print("=" * 70)
print(f"\n✅ Generated {len(embeddings)} embeddings")
print(f"📊 Each embedding has {embeddings[0].shape[0]} dimensions\n")

# Show first few dimensions of each embedding
for i, (sentence, embedding) in enumerate(zip(sentences, embeddings)):
    print(f"\nSentence {i+1}: '{sentence}'")
    print(f"Embedding (first 10 values): {embedding[:10]}")
    print(f"Embedding shape: {embedding.shape}")

### 🔍 Comparing Sentence Similarities

Now let's use our distance functions to see which sentences are most similar!

In [ ]:
# Use the first sentence as our query
query_idx = 0
query_embedding = embeddings[query_idx]
query_sentence = sentences[query_idx]

print("=" * 70)
print("SENTENCE SIMILARITY ANALYSIS")
print("=" * 70)
print(f"\n🔍 Query Sentence: '{query_sentence}'\n")
print("-" * 70)

# Compute similarities with all other sentences
results = []
for i, (sentence, embedding) in enumerate(zip(sentences, embeddings)):
    if i == query_idx:
        continue

    cos_sim = cosine_similarity(query_embedding, embedding)[0]
    euc_dist = euclidean_distance(query_embedding, embedding)[0]

    results.append({
        'index': i,
        'sentence': sentence,
        'cosine_similarity': cos_sim,
        'euclidean_distance': euc_dist
    })

# Sort by cosine similarity (descending)
results_by_cosine = sorted(results, key=lambda x: x['cosine_similarity'], reverse=True)

print("RANKED BY COSINE SIMILARITY (Higher = More Similar):")
print("-" * 70)
for rank, result in enumerate(results_by_cosine, 1):
    marker = "⭐" if rank == 1 else "  "
    print(f"{marker} Rank {rank}: {result['sentence']}")
    print(f"   Cosine Similarity: {result['cosine_similarity']:.4f}")
    print(f"   Euclidean Distance: {result['euclidean_distance']:.4f}\n")

print("=" * 70)
print("💡 INTERPRETATION")
print("=" * 70)
print("The sentence with the highest cosine similarity is the most")
print("semantically similar to our query sentence.")
print("\nNotice how semantically similar sentences (like cat/feline)")
print("have higher similarity scores than unrelated sentences!")
print("=" * 70)

## 2.4 Visualizing Word Embeddings

Embeddings exist in **768-dimensional space**, which is impossible to visualize directly. We'll use **PCA (Principal Component Analysis)** to reduce them to 2D while preserving as much information as possible.

### What is PCA?

PCA finds the directions of maximum variance in high-dimensional data and projects the data onto a lower-dimensional space along those directions.

In [ ]:
def visualize_embeddings(sentences, embeddings):
    """
    Visualize high-dimensional embeddings in 2D using PCA.

    Parameters:
    -----------
    sentences : list of str
        The original sentences
    embeddings : numpy array
        The embedding vectors (shape: n_sentences x embedding_dim)
    """
    # Reduce dimensions from 768 to 2 using PCA
    pca = PCA(n_components=2)
    embeddings_2d = pca.fit_transform(embeddings)

    # Create the visualization
    plt.figure(figsize=(14, 10))

    # Define colors for each sentence
    colors = ['blue', 'green', 'red', 'purple', 'orange']

    # Plot each embedding
    for i, (x, y) in enumerate(embeddings_2d):
        plt.scatter(x, y, c=colors[i], s=200, alpha=0.6, edgecolors='black', linewidth=2)

        # Add sentence labels with word wrapping
        label = sentences[i]
        if len(label) > 40:
            # Wrap long sentences
            words = label.split()
            label = '\n'.join([' '.join(words[j:j+5]) for j in range(0, len(words), 5)])

        plt.annotate(label, (x, y),
                    xytext=(10, 10),
                    textcoords='offset points',
                    bbox=dict(boxstyle='round,pad=0.5', facecolor=colors[i], alpha=0.3),
                    fontsize=9,
                    ha='left')

    plt.title('2D Visualization of Sentence Embeddings (using PCA)', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel(f'First Principal Component ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=12)
    plt.ylabel(f'Second Principal Component ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=12)
    plt.grid(True, alpha=0.3)

    # Add explanation text
    explanation = (
        f"Note: Original embeddings are {embeddings.shape[1]}-dimensional.\n"
        f"PCA reduces to 2D for visualization, preserving {pca.explained_variance_ratio_.sum()*100:.1f}% of variance.\n"
        "Points closer together represent more semantically similar sentences."
    )
    plt.figtext(0.5, 0.02, explanation, ha='center', fontsize=10,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()
    plt.show()

    # Print PCA information
    print("\n" + "=" * 70)
    print("PCA DIMENSIONALITY REDUCTION INFO")
    print("=" * 70)
    print(f"Original dimensions: {embeddings.shape[1]}")
    print(f"Reduced dimensions: 2")
    print(f"Variance explained by PC1: {pca.explained_variance_ratio_[0]*100:.2f}%")
    print(f"Variance explained by PC2: {pca.explained_variance_ratio_[1]*100:.2f}%")
    print(f"Total variance preserved: {pca.explained_variance_ratio_.sum()*100:.2f}%")
    print("=" * 70)

# Visualize the embeddings
visualize_embeddings(sentences, embeddings)

---

# 3 - Embeddings and Input Size

In this section, we explore how the **size of input text** impacts the generation of vector embeddings.

## 🔑 Key Concepts:

- **Vector embeddings** capture the semantic essence of a text
- There is a **limit** to how much text these models can process at once
- Text exceeding this limit gets **truncated**
- When truncation occurs, all information beyond a certain point is **lost**

### ⚠️ Impact of Truncation:

Truncation can potentially impact:
- Effectiveness of the embedding
- Accuracy of similarity comparisons
- Overall performance in RAG systems

## 3.1 An Example

The example below illustrates how semantic meaning can be lost due to truncation. Any information beyond a certain point in the input text is not considered, leading to an **incomplete representation** in the embedding vector.

Let's create a large text and see what happens!

In [ ]:
# Create a large text file for demonstration
large_text_content = """
The Evolution of Artificial Intelligence: A Comprehensive Overview

Chapter 1: Introduction to Artificial Intelligence

Artificial Intelligence (AI) has become one of the most transformative technologies of the 21st century.
From its humble beginnings in the 1950s to the sophisticated systems we see today, AI has revolutionized
how we interact with technology, process information, and solve complex problems.

The field of AI encompasses various subdomains including machine learning, natural language processing,
computer vision, robotics, and expert systems. Each of these areas has contributed significantly to the
advancement of intelligent systems capable of performing tasks that traditionally required human intelligence.

Chapter 2: Machine Learning Fundamentals

Machine learning, a subset of AI, focuses on developing algorithms that can learn from and make predictions
or decisions based on data. Unlike traditional programming where explicit instructions are provided, machine
learning systems improve their performance through experience.

There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement
learning. Supervised learning involves training models on labeled data, where the correct output is known.
Unsupervised learning deals with unlabeled data, seeking to find hidden patterns or structures. Reinforcement
learning involves agents learning to make decisions by receiving rewards or penalties for their actions.

Chapter 3: Deep Learning Revolution

Deep learning, inspired by the structure and function of the human brain, has led to breakthrough achievements
in AI. Neural networks with multiple layers (hence "deep") can learn hierarchical representations of data,
enabling them to tackle complex tasks such as image recognition, speech synthesis, and language translation.

The success of deep learning can be attributed to several factors: the availability of large datasets,
increased computational power through GPUs, and algorithmic innovations such as backpropagation and various
optimization techniques.

Chapter 4: Natural Language Processing

Natural Language Processing (NLP) enables computers to understand, interpret, and generate human language.
Recent advances in NLP, particularly with transformer architectures like BERT and GPT, have dramatically
improved the quality of language understanding and generation.

Applications of NLP include sentiment analysis, machine translation, question answering, text summarization,
and chatbots. These technologies are now ubiquitous in our daily lives, powering virtual assistants,
search engines, and content recommendation systems.

Chapter 5: Computer Vision

Computer vision enables machines to interpret and understand visual information from the world. Through
convolutional neural networks (CNNs) and other advanced techniques, computers can now perform tasks such
as object detection, facial recognition, image segmentation, and video analysis with remarkable accuracy.

The applications of computer vision span numerous industries including healthcare (medical imaging),
automotive (autonomous vehicles), retail (visual search), and security (surveillance systems).

Chapter 6: Ethical Considerations and Future Directions

As AI systems become more powerful and pervasive, ethical considerations have come to the forefront.
Issues such as bias in AI systems, privacy concerns, job displacement, and the potential misuse of AI
technologies require careful consideration and proactive governance.

The future of AI holds immense promise, with ongoing research in areas such as artificial general intelligence,
quantum machine learning, and neuromorphic computing. As we continue to push the boundaries of what's possible,
it's crucial that we develop AI systems that are not only powerful but also safe, fair, and beneficial to all
of humanity.

THIS IS CRITICAL INFORMATION AT THE END: The most important conclusion of this document is that AI must be
developed responsibly with human values at its core. The final recommendation is to establish international
cooperation frameworks for AI governance and to prioritize AI safety research. This concluding section
contains the key takeaways and action items that should not be missed.
"""

# Save to file in Colab's working directory
file_path = "/content/large_text.txt"

with open(file_path, "w") as f:
    f.write(large_text_content)

print("✅ Large text file created!")
print(f"📄 File size: {len(large_text_content)} characters")
print(f"📝 Approximate word count: {len(large_text_content.split())} words")
print(f"📂 Saved at: {file_path}")

In [ ]:
# Load the large text
with open('/content/large_text.txt', 'r') as f:
    big_text = f.read()

print("=" * 70)
print("LARGE TEXT ANALYSIS")
print("=" * 70)
print(f"\n📊 Text Statistics:")
print(f"   Total characters: {len(big_text)}")
print(f"   Total words: {len(big_text.split())}")
print(f"   Total lines: {len(big_text.splitlines())}")
print("\n📝 First 500 characters:")
print("-" * 70)
print(big_text[:500] + "...")
print("-" * 70)

### 🧪 Testing the Truncation Effect

Now let's generate embeddings for:
1. The **entire text**
2. Only the **first 3000 characters**

If the model has a context window limit, these should produce **identical embeddings**!

In [ ]:
print("Generating embeddings...\n")

# Entire text
big_text_embedding = model.encode(big_text)

# Text with fewer characters (first 3000)
big_text_embedding_few_characters = model.encode(big_text[:3000])

# Check if they are the same
are_equal = np.array_equal(big_text_embedding, big_text_embedding_few_characters)

print("=" * 70)
print("TRUNCATION TEST RESULTS")
print("=" * 70)
print(f"\n📊 Embedding Comparison:")
print(f"   Full text length: {len(big_text)} characters")
print(f"   Truncated text length: 3000 characters")
print(f"   Embedding dimension: {len(big_text_embedding)}")
print(f"\n🔍 Are the embeddings identical? {are_equal}")

if are_equal:
    print("\n⚠️  WARNING: Truncation Detected!")
    print("-" * 70)
    print("The embeddings are IDENTICAL, which means:")
    print("")
    print("1. The model has a context window limit (512 tokens for this model)")
    print("2. Everything beyond that limit is COMPLETELY IGNORED")
    print("3. Important information at the end of the text is LOST")
    print("")
    print("💡 In this example, the critical conclusion about AI governance")
    print("   and safety recommendations at the END of the document were")
    print("   completely ignored by the embedding model!")
else:
    print("\n✅ No truncation detected - embeddings are different")

print("\n" + "=" * 70)
print("ELEMENT-WISE COMPARISON")
print("=" * 70)
print(f"Number of different elements: {np.sum(big_text_embedding != big_text_embedding_few_characters)}")
print(f"Maximum absolute difference: {np.max(np.abs(big_text_embedding - big_text_embedding_few_characters)):.10f}")
print("=" * 70)

### 📚 Understanding the Results

Note that they are the **same vector** - not even a single element different!

#### Why does this happen?

This is because the **context window** for this model is **512 tokens**, so anything beyond that is completely ignored.

#### What does this mean?

- ⚠️ **Loss of Information**: Important content at the end of long documents is not captured
- ⚠️ **Incomplete Representations**: The embedding doesn't represent the full document
- ⚠️ **RAG Limitations**: In retrieval systems, this can lead to missing relevant information

#### 💡 Solutions (covered in future modules):

1. **Text Chunking**: Break large documents into smaller chunks
2. **Sliding Windows**: Use overlapping windows to capture all content
3. **Hierarchical Embeddings**: Create embeddings at multiple levels
4. **Summarization**: Generate concise summaries before embedding

In the **next modules**, you will learn advanced techniques to handle large texts effectively!

## 🔬 Additional Experiments

Let's explore a few more scenarios to deepen our understanding:

In [ ]:
# Experiment: Semantic similarity with paraphrases
original = "The quick brown fox jumps over the lazy dog."
paraphrase = "A fast brown fox leaps over a sleepy dog."
unrelated = "Machine learning models require large datasets for training."

# Generate embeddings
emb_original = model.encode(original)
emb_paraphrase = model.encode(paraphrase)
emb_unrelated = model.encode(unrelated)

# Calculate similarities
sim_paraphrase = cosine_similarity(emb_original, emb_paraphrase)[0]
sim_unrelated = cosine_similarity(emb_original, emb_unrelated)[0]

print("=" * 70)
print("SEMANTIC SIMILARITY EXPERIMENT")
print("=" * 70)
print(f"\nOriginal: '{original}'")
print(f"\nParaphrase: '{paraphrase}'")
print(f"Similarity: {sim_paraphrase:.4f} ⭐")
print(f"\nUnrelated: '{unrelated}'")
print(f"Similarity: {sim_unrelated:.4f}")
print("\n💡 Notice how the paraphrase has MUCH higher similarity")
print("   despite using different words! This shows the model")
print("   understands semantic meaning, not just word matching.")
print("=" * 70)

---

# 4 - Conclusion

## 🎉 Congratulations!

You have successfully completed the lab on **Vector Embeddings in RAG**!

### 📚 What You've Learned:

1. ✅ **Vector Embeddings Fundamentals**
   - How embeddings capture semantic meaning
   - The role of embeddings in RAG systems

2. ✅ **Distance Metrics**
   - Cosine Similarity (angle-based)
   - Euclidean Distance (magnitude-based)
   - When to use each metric

3. ✅ **Practical Implementation**
   - Loading and using transformer-based embedding models
   - Generating embeddings for real text
   - Comparing semantic similarities

4. ✅ **Visualization Techniques**
   - Using PCA for dimensionality reduction
   - Visualizing high-dimensional embeddings

5. ✅ **Input Size Limitations**
   - Understanding context window limits
   - Impact of truncation on embeddings
   - Preview of advanced techniques for handling large texts

### 🚀 Next Steps:

In upcoming modules, you will learn:
- Advanced chunking strategies for large documents
- Building complete RAG systems
- Vector databases and efficient retrieval
- Hybrid search approaches
- Production deployment considerations

### 💪 Keep it up!

You're building a solid foundation in vector embeddings and RAG systems. These skills are crucial for modern AI applications!

---

## 📝 Key Takeaways:

```
┌─────────────────────────────────────────────────────────────┐
│  1. Embeddings convert text to numerical vectors           │
│  2. Similar meanings → Similar vectors                     │
│  3. Distance metrics measure similarity                    │
│  4. Context windows limit input size                       │
│  5. Proper handling of large texts is crucial             │
└─────────────────────────────────────────────────────────────┘
```

---

### 🙏 Thank you for completing this lab!

**Author**: Adapted for Google Colab  
**Original Source**: DeepLearning.AI  
**Model Used**: BAAI/bge-base-en-v1.5  
**Date**: 2026

## 📖 References and Further Reading

1. **Sentence Transformers Documentation**  
   https://www.sbert.net/

2. **BAAI/bge-base-en-v1.5 Model Card**  
   https://huggingface.co/BAAI/bge-base-en-v1.5

3. **Understanding Vector Embeddings**  
   https://towardsdatascience.com/understanding-vector-embeddings

4. **RAG Systems Overview**  
   https://arxiv.org/abs/2005.11401

5. **PCA for Dimensionality Reduction**  
   https://scikit-learn.org/stable/modules/decomposition.html

---



In [ ]:
import json
import os

# 1. This tells Colab to internally fix the metadata of the current session
from google.colab import drive

def inject_github_friendly_metadata():
    # In Colab, the notebook JSON is handled in memory,
    # but we can force the session to adopt this structure.
    metadata_patch = {
        "widgets": {
            "application/vnd.jupyter.widget-state+json": {
                "state": {}
            }
        }
    }

    # We display this to confirm what GitHub needs to see
    print("Metadata fix prepared. GitHub requires the 'state' key inside 'widgets'.")
    return metadata_patch

inject_github_friendly_metadata()